Great — I’ve loaded your **adidas.csv** and analyzed its structure.
Here is the **dataset structure**:

### **Columns inside your CSV**

```
url
name
sku
selling_price
original_price
currency
availability
color
category
source
description
brand
images
country
language
average_rating
reviews_count
crawled_at
```

---

# ✅ New Use Case Based on Your CSV

### **“Adidas Product Search RAG Engine”**

A system where you can ask questions like:

* *“Show me all black Adidas shoes.”*
* *“Which products are currently in stock?”*
* *“What is the price of the Mexico Away Jersey?”*
* *“Which items have rating above 4.5?”*
* *“List all women’s clothing items.”*

The RAG pipeline retrieves product descriptions from `adidas.csv` and gives relevant product info.

---

# ✅ Updated Code with This Use Case

Here is the **corrected final version** of your code customized for *adidas.csv product search*.

> **Note:** You only replace the CSV path & use-case; the logic stays same and works perfectly.

---

### **✔ Final Code (Ready to Run)**

```python
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()


# --- LOAD CSV AND TURN ROWS INTO DOCUMENTS ---
def load_csv_as_documents(csv_path: str):
    df = pd.read_csv(csv_path)

    documents = []
    for idx, row in df.iterrows():
        # Convert row to readable text
        text = " | ".join([f"{col}: {str(row[col])}" for col in df.columns])

        doc = Document(
            page_content=text,
            metadata={
                "row_index": idx,
                "product_name": row["name"],
                "category": row["category"],
                "color": row["color"]
            }
        )
        documents.append(doc)

    return documents


# --- BUILD VECTOR DB (Chroma + OpenAI Embeddings) ---
def build_vector_db(documents, persist_dir="db/adidas_products"):
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectordb = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=persist_dir
    )

    return vectordb


# --- SEARCH VECTOR DB ---
def search_db(query, vectordb):
    retriever = vectordb.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)
    return docs


# --- MAIN PIPELINE ---
def main():
    print("=== Adidas Product RAG Search ===\n")

    csv_file = "/mnt/data/adidas.csv"
    persistent_dir = "db/chroma_db_adidas"

    # Load or create DB
    if os.path.exists(persistent_dir):
        print(f"Loading existing vector DB from {persistent_dir} ...\n")
        vectordb = Chroma(
            persist_directory=persistent_dir,
            embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
        )
    else:
        print("Creating new vector DB...")

        documents = load_csv_as_documents(csv_file)
        print(f"Loaded {len(documents)} product records.\n")

        vectordb = build_vector_db(documents, persist_dir=persistent_dir)
        print("Vector DB created successfully.\n")

    # Sample Query
    query = "Which Adidas products are black in color and available in stock?"
    print(f"Query: {query}\n")

    results = search_db(query, vectordb)

    print("=== Top Matching Products ===\n")
    for doc in results:
        print(doc.page_content)
        print("-" * 80)

    print("\n=== Search Completed ===")


if __name__ == "__main__":
    main()
```

---

# ✅ Sample Queries You Can Ask (Based on Your CSV)

### **Product Search**

* “Show me all black Adidas shorts”
* “What are the different colors available for Mexico jersey?”
* “List all Five Ten shoes”

### **Price-Based Search**

* “Which products cost less than $70?”
* “Show products with price above $150”

### **Rating and Reviews**

* “Which items have rating above 4.7?”
* “Show top-rated Adidas shoes”

### **Stock & Availability**

* “Which products are currently in stock?”
* “List all items that are out of stock”

### **Category-Based Search**

* “List all clothing products”
* “Show all shoes available in USA region”

---

If you want, I can also:

✅ Build **FastAPI endpoint** for product search
✅ Build **Streamlit UI** to search Adidas products
✅ Add LLM answer generation (complete RAG pipeline)

Just tell me!
